In [1]:
import numpy as np
from multiprocessing import Pool
import gutpython
import os

In [2]:
initial_throwaway_time = 350

In [3]:
%matplotlib ipympl

In [4]:
default_numerical_param_dict = dict(
    max_stuck_chance=50,
    low_stuck_bound=2,
    unstuck_chance=10.0,
    mid_stuck_conc=10.0,
    seed_chance=5.0,
    seed_percent=5.0,
    absorption=0.0,
    reserve_fraction=0.0,
    bifido_lactate_production=0.005,
    flow_dist=0.28,
    bifido_doub=330,
    desulfo_doub=330,
    bacteroid_doub=330,
    clost_doub=330,
    # initialization constants
    init_num_bifidos=23562,
    init_num_bacteroids=5490,
    init_num_closts=921,
    init_num_desulfos=70,
)

In [5]:
def numerical_param_sample():
    proportions = np.exp(np.log(2) * np.random.randn(len(default_numerical_param_dict)))
    return {
        param_name: int(prop * param_value) if isinstance(param_value, int) else float(prop * param_value)
        for prop, (param_name, param_value) in zip(proportions, default_numerical_param_dict.items())
    }

In [6]:
def control_param_sample():
    control_params = {}
    p = 0.1

    if np.random.random() < p:
        control_params['in_conc_bacteroids'] = lambda t: 10 if t < initial_throwaway_time + 100 else 0
    else:
        control_params['in_conc_bacteroids'] = lambda t: 0

    if np.random.random() < p:
        control_params['in_conc_bifidos'] = (lambda t: 10 if t < initial_throwaway_time + 100 else 0) 
    else:
        control_params['in_conc_bifidos'] = lambda t: 0

    if np.random.random() < p:
        control_params['in_conc_closts'] = (lambda t: 10 if t < initial_throwaway_time + 100 else 0)
    else:
        control_params['in_conc_closts'] = lambda t: 0

    if np.random.random() < p:
        control_params['in_conc_desulfos'] = (lambda t: 10 if t < initial_throwaway_time + 100 else 0)
    else:
        control_params['in_conc_desulfos'] = lambda t: 0

    if np.random.random() < p:
        control_params['cs_inflow'] = (lambda t: 0.2 if t < initial_throwaway_time + 100 else 0.1) 
    else:
        control_params['cs_inflow'] = lambda t: 0.1

    if np.random.random() < p:
        control_params['fo_inflow']=  (lambda t: 50.0 if t < initial_throwaway_time + 100 else 25.0)
    else:
        control_params['fo_inflow']= lambda t: 25.0

    if np.random.random() < p:
        control_params['glucose_inflow'] =  (lambda t: 60.0 if t < initial_throwaway_time + 100 else 30.0)
    else:
        control_params['glucose_inflow']=lambda t: 30.0

    if np.random.random() < p:
        control_params['inulin_inflow']= (lambda t: 20.0 if t < initial_throwaway_time + 100 else 10.0)
    else:
        control_params['inulin_inflow']=lambda t: 10.0

    if np.random.random() < p:
        control_params['lactose_inflow']= (lambda t: 30.0 if t < initial_throwaway_time + 100 else 15.0)
    else:
        control_params['lactose_inflow']=lambda t: 15.0

    if np.random.random() < p:
        control_params['lactate_inflow']= (lambda t: 1.0 if t < initial_throwaway_time + 100 else 0.0)
    else:
        control_params['lactate_inflow']=lambda t: 0.0
        
    return control_params

In [8]:
def make_model(numerical_param_dict, control_param_dict):
    model = gutpython.GutPython(
        **numerical_param_dict,
        **control_param_dict,
        tick_in_flow=480,
    )
    model.setup()
    for t in range(initial_throwaway_time):
        model.go()

    return model

In [11]:
data_storage_dir = "/big-data/knappa/gutpython/sims"

def take_sample(sample_idx):
    model = make_model(numerical_param_sample(), control_param_sample())
    filename = os.path.join(data_storage_dir, f"sample-{str(sample_idx).zfill(4)}.hdf5")

    model.save(filename, write_mode='a')
    for t in range(initial_throwaway_time, initial_throwaway_time + 500):
        model.go()/big-data/knappa/gutpython/sims
        model.save(filename, write_mode='a')

In [ ]:
with Pool(10) as p:
    p.map(take_sample, range(100))